# Autism Image Classification

This notebook lets you try a model trained to distinguish photos of autistic and non-autistic children using the area around their eyes. It was trained on photos of children, not adults. It’s a research example, not a screening or diagnostic tool.

**New to Colab?** Each block of code is called a **cell**. Click the ▶ button on the left to run it. A spinning circle means the cell is still running. A green check on the left means it finished successfully. You can use the checkboxes and menus without editing the code.

**Where to start:** Run **Cells 1 and 2** first. To try a photo, go to **Cell 5** and choose **Saved project model**, or choose **Retrained in this runtime** if you trained a model in **Cell 3**. To see how either model performs on the included test set, go to **Cell 4** and select the model you want to evaluate.


## Cell 1: Load the project

Run this cell to upload the project ZIP. When Colab asks you to choose a file, select the same **autism-image-classification.zip** you downloaded. Don’t select the notebook you just opened or a separate ZIP of photos. Wait for the notebook to confirm the project is loaded.


In [ ]:
# @title Cell 1. Load the project
LOAD_TRUSTED_PROJECT = True  # @param {type:"boolean"}

import os
from hashlib import sha256
from pathlib import Path, PurePosixPath
import shutil
import stat
import subprocess
import sys
import tempfile
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

WORK = Path("/content/autism-image-classification-runtime")
SCRIPTS = ("prepare_dataset", "preprocess", "train", "predict", "bayes",
           "features", "model", "crop", "test_inputs", "training_inputs", "analyze")
ARTIFACTS = ("pca.joblib", "scaler.joblib", "confusion_matrix.joblib")
MODEL_FILENAMES = ("pretrained_model.pt", "Learned Model Full", "Learned Model Full.zip")
PHOTO_TYPES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def checked_extract(archive_path, destination):
    """Unpack a project archive without allowing path escapes or overwrites."""
    destination.mkdir()
    with ZipFile(archive_path) as archive:
        members = archive.infolist()
        if len(members) > 25_000 or sum(m.file_size for m in members) > 2 * 1024**3:
            raise ValueError("This archive exceeds the project loader's size limit.")
        for member in members:
            path = PurePosixPath(member.filename.replace(chr(92), "/"))
            mode = stat.S_IFMT(member.external_attr >> 16)
            if (path.is_absolute() or not path.parts or ".." in path.parts
                    or any(":" in part for part in path.parts)
                    or mode not in (0, stat.S_IFREG, stat.S_IFDIR)):
                raise ValueError("The project ZIP contains an unsafe path or file type.")
            target = destination.joinpath(*path.parts)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)


def find_project_root(folder):
    roots = [
        req.parent for req in folder.rglob("requirements-photos.txt")
        if (req.parent / "requirements.txt").is_file()
        and all((req.parent / "scripts" / f"{name}.py").is_file() for name in SCRIPTS)
    ]
    if len(roots) != 1:
        raise ValueError("Choose a ZIP containing one complete project.")
    saved = roots[0] / "artifacts" / "pretrained"
    missing = [name for name in ARTIFACTS if not (saved / name).is_file()]
    if not any((saved / name).is_file() for name in MODEL_FILENAMES):
        missing.append(MODEL_FILENAMES[0])
    if missing:
        raise FileNotFoundError("The project is missing saved artifacts: " + ", ".join(missing))
    return roots[0]


def project_root():
    if "REPO" not in globals() or not REPO.is_dir():
        raise RuntimeError("First run Cell 1 with LOAD_TRUSTED_PROJECT checked.")
    return REPO


def bundled_test_archive():
    folder = project_root() / "data" / "evaluation"
    archive = folder / "test-eye-strips.zip"
    if not archive.is_file() and (folder / "Test.zip").is_file():
        archive = folder / "Test.zip"
    if not archive.is_file():
        raise FileNotFoundError("The bundled Test archive is missing. "
                                "Load the full project ZIP in Cell 1, then rerun this cell.")
    return archive


def bundled_processed_archive():
    folder = project_root() / "data"
    archive = folder / "processed-eye-strips.zip"
    if not archive.is_file() and (folder / "Processed_Data_Eye_Strip.zip").is_file():
        archive = folder / "Processed_Data_Eye_Strip.zip"
    if not archive.is_file():
        raise FileNotFoundError("The bundled processed archive is missing. "
                                "Load the full project ZIP in Cell 1, then rerun this cell.")
    return archive

def bayes_model_identity(source):
    """Identify the exact network, PCA and scaler used for rates or predictions."""
    if source == "Saved project model":
        files_dir = SAVED
        model_file = next((files_dir / name for name in MODEL_FILENAMES
                           if (files_dir / name).is_file()),
                          files_dir / MODEL_FILENAMES[0])
    elif source == "Retrained in this runtime":
        files_dir = globals().get("TRAINED")
        if files_dir is None:
            raise RuntimeError("Retrain a model in Cell 3, then evaluate it in Cell 4.")
        model_file = files_dir / "model.pt"
    else:
        raise ValueError("Choose one of the listed models.")
    fingerprints = []
    for path in (model_file, files_dir / "pca.joblib", files_dir / "scaler.joblib"):
        if not path.is_file():
            if source == "Retrained in this runtime":
                raise RuntimeError("Retrain a model in Cell 3, then evaluate it in Cell 4.")
            raise RuntimeError("Load the full saved project ZIP in Cell 1.")
        fingerprints.append(sha256(path.read_bytes()).hexdigest())
    return source, files_dir.resolve(), tuple(fingerprints)


def invalidate_bayesian_controls():
    global BAYES_INPUT_GENERATION
    BAYES_INPUT_GENERATION = object()
    globals().get("RESULTS", {}).pop("Bayesian example (ZIP)", None)


# Invalidate Cell 6 defaults and any earlier controls when a project is loaded again.
PREDICTION_COUNTS = None
PREDICTION_MODEL_ID = None
RETRAINED_BAYES_METRICS = None
invalidate_bayesian_controls()

if LOAD_TRUSTED_PROJECT:
    WORK.mkdir(exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix="project-", dir=WORK))
    source = stage / "source"
    print("Choose autism-image-classification.zip, not a ZIP of photos or data.")
    archive = stage / "project.zip"
    files.upload_file(str(archive))
    checked_extract(archive, source)
    archive.unlink()
    REPO = find_project_root(source)
    SAVED = REPO / "artifacts" / "pretrained"
    TRAINED = None
    RESULTS = {}
    print("Project loaded. Continue to Cell 2.")
else:
    print("Nothing loaded. Check LOAD_TRUSTED_PROJECT to start.")

## Cell 2: Set up Colab

Run this cell and wait for it to finish. It installs what the project needs in Colab, not on your computer. This can take a few minutes.

You only need to run it once per session. If you start a new Colab session later, run **Cells 1 and 2** again. To try a photo now, jump to **Cell 5**.


In [ ]:
# @title Cell 2. Install the project environment
import os
os.environ["MPLBACKEND"] = "Agg"

INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

def run_command(args, **kwargs):
    return subprocess.run([str(value) for value in args], check=True, **kwargs)


def runtime_python():
    if not globals().get("ENV_READY", False):
        raise RuntimeError("First run Cell 2 with INSTALL_DEPENDENCIES checked.")
    return VENV_PYTHON


def run_script(script, *args, private=False):
    repo = project_root()
    command = [runtime_python(), repo / "scripts" / f"{script}.py", *args]
    options = {"cwd": repo, "env": dict(os.environ, MPLBACKEND="Agg")}
    if not private:
        return run_command(command, **options)
    result = subprocess.run([str(value) for value in command], capture_output=True, text=True, **options)
    if result.returncode:
        raise RuntimeError("Prediction did not complete. Check that the photos are readable and "
                           "show a face with open eyes and no glasses. If needed, enable "
                           "SHOW_PREDICTION_LOG; the log can include photo filenames.")
    return result


def run_eye_crop(source, output, *, show_log=False):
    """Crop uploaded photos in Colab’s notebook Python."""
    global NATIVE_CROP_READY
    repo = project_root()
    env = dict(os.environ, MPLBACKEND="Agg")
    if not globals().get("NATIVE_CROP_READY", False):
        for dependency in ("mediapipe", "glasses-detector"):
            run_command([sys.executable, "-m", "pip", "-q", "install", dependency], env=env)
        NATIVE_CROP_READY = True

    result = subprocess.run(
        [sys.executable, str(repo / "scripts" / "crop.py"), str(source), str(output)],
        cwd=repo, env=env, capture_output=True, text=True,
    )
    if result.returncode:
        detail = result.stderr.strip() if show_log else ""
        raise RuntimeError(detail or "Eye cropping failed. Check the cropper dependencies and inputs.")
    if show_log:
        print(result.stdout.strip())
    else:
        for line in result.stdout.splitlines():
            if line.startswith(("Found ", "Crops saved:", "Images with no detected face:",
                                "Unreadable/failed images:")):
                print(line)


if INSTALL_DEPENDENCIES:
    repo = project_root()
    ENV_READY = False
    venv = WORK / "venv-py312"
    VENV_PYTHON = venv / "bin" / "python"
    if not VENV_PYTHON.exists():
        run_command([sys.executable, "-m", "pip", "install", "uv"])
        run_command([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", venv])
    run_command([VENV_PYTHON, "-c", "import sys; assert sys.version_info[:2] == (3, 12)"])
    run_command([VENV_PYTHON, "-m", "pip", "install", "torch==2.11.0", "torchvision==0.26.0",
                 "--index-url", "https://download.pytorch.org/whl/cpu"])
    run_command([VENV_PYTHON, "-m", "pip", "install", "-r", repo / "requirements-photos.txt",
                 "scikit-learn==1.6.1", "jax==0.6.2", "jaxlib==0.6.2"])
    run_command([VENV_PYTHON, "-m", "pip", "check"])
    run_command([VENV_PYTHON, "-c",
                 "import cv2, mediapipe, glasses_detector, sklearn, torch, torchvision; "
                 "assert sklearn.__version__ == '1.6.1'; "
                 "print('Python 3.12 environment and key imports verified.')"])
    ENV_READY = True
else:
    print("Installation skipped. Check INSTALL_DEPENDENCIES when ready.")

## Cell 3: Train a new model (optional)

Skip this cell to use the model already included with the project. To train a new one, check the retraining box and run the cell. It uses the cropped Training and Validation images included in the project ZIP. No extra upload is needed.

Keep the default settings to use the same model size as the included model, or adjust the settings below before running the cell:

- **PCA components (`PCA_COMPONENTS`, default: 17):** how many image patterns the model uses to represent each eye crop. PCA is short for **principal component analysis**.
- **Hidden-layer neurons (`HIDDEN_NEURONS`, default: 60):** how many units the neural network uses to combine those inputs.
- **Maximum epochs (`MAXIMUM_EPOCHS`, default: 400):** the maximum number of passes through the training data. One pass is called an epoch.
- **Patience (`PATIENCE`, default: 50):** how many epochs to wait without improvement on the validation set before stopping early. If validation loss (a measure of prediction error) improves, the count starts over. Training keeps the version with the lowest validation loss.

The model learns from the **training set**. The **validation set** helps the notebook choose which version of the model to keep. The **test set** is held back until Cell 4.

When training finishes, choose **Retrained in this runtime** in the cells where you want to use the new model. A **runtime** is the temporary Colab session running this notebook.


In [ ]:
# @title Cell 3. Optionally retrain the model
RUN_RETRAINING = False  # @param {type:"boolean"}
PCA_COMPONENTS = 17  # @param {type:"integer"}
HIDDEN_NEURONS = 60  # @param {type:"integer"}
MAXIMUM_EPOCHS = 400  # @param {type:"integer"}
PATIENCE = 50  # @param {type:"integer"}

if RUN_RETRAINING:
    invalidate_bayesian_controls()
    RETRAINED_BAYES_METRICS = None
    # Do not offer an older run as the result of a failed or interrupted retraining.
    TRAINED = None
    repo = project_root()
    runtime_python()
    RESULTS.pop("Retrained model and training log (ZIP)", None)
    settings = {"PCA_COMPONENTS": PCA_COMPONENTS, "HIDDEN_NEURONS": HIDDEN_NEURONS,
                "MAXIMUM_EPOCHS": MAXIMUM_EPOCHS, "PATIENCE": PATIENCE}
    for name, value in settings.items():
        if type(value) is not int or value < 1:
            raise ValueError(f"{name} must be a positive whole number.")

    fixed_test = bundled_test_archive()
    with tempfile.TemporaryDirectory(prefix="training-", dir=WORK) as folder:
        stage = Path(folder)
        crops = stage / "crops"
        print("Using the processed Training and Validation strips bundled with the project.")
        run_script("training_inputs", "cropped", bundled_processed_archive(), crops)
        # The staging script extracts only usable file types; each strip has 128 x 20 pixels.
        training_images = sum(path.is_file() for path in (crops / "Training Data").rglob("*"))
        max_components = min(training_images, 128 * 20)
        if PCA_COMPONENTS > max_components:
            raise ValueError(
                f"PCA_COMPONENTS must be at most {max_components:,} for these training data. "
                "Lower PCA_COMPONENTS and rerun Cell 3."
            )

        run_script("training_inputs", "check-holdout", crops, fixed_test)

        retrained_root = repo / "outputs" / "retrained"
        retrained_root.mkdir(parents=True, exist_ok=True)
        result = Path(tempfile.mkdtemp(prefix="run-", dir=retrained_root))
        run_script("train", "--fit-only", "--data-dir", crops, "--artifacts-dir", result,
                   "--output-dir", result, "--epochs", MAXIMUM_EPOCHS, "--patience", PATIENCE,
                   "--components", PCA_COMPONENTS, "--hidden-units", HIDDEN_NEURONS,
                   "--positive-class", "Autistic", "--negative-class", "Non-Autistic",
                   "--skip-sensitivity")

    names = ("model.pt", "pca.joblib", "scaler.joblib", "training_metrics.json")
    artifacts = tuple(result / name for name in names)
    if not all(path.is_file() for path in artifacts):
        raise RuntimeError("Training did not produce every expected artifact.")
    TRAINED = result
    RESULTS["Retrained model and training log (ZIP)"] = artifacts
    print("Training complete. Select the retrained model in Cell 4 or 5 to use it.")
    print("No Test results were calculated during training. Download artifacts in Cell 7.")
else:
    print("Retraining skipped.")

## Cell 4: Check model performance

Choose **Saved project model** or **Retrained in this runtime**, then run the cell. No upload is needed. Both models are evaluated on the same included test set, which was not used to train the model or choose which version to keep.

The **confusion matrix** compares the model’s predictions with the dataset labels on those test photos. **Accuracy** is the share classified correctly. **Sensitivity** is the share of autistic-labeled photos correctly identified. **Specificity** is the share of non-autistic-labeled photos correctly identified.

If you enable the **pixel-sensitivity heatmap**, it shows where small changes in brightness around the eyes tend to affect the model’s score most. It does not show which features cause autism.


In [ ]:
# @title Cell 4. Evaluate the model on the bundled Test set
RUN_MODEL_ANALYSIS = True  # @param {type:"boolean"}
ANALYSIS_MODEL = "Saved project model"  # @param ["Saved project model", "Retrained in this runtime"]
RUN_PIXEL_SENSITIVITY = True  # @param {type:"boolean"}

import json
import pandas as pd
from IPython.display import Image, display


def display_test_results(output):
    metrics_file = output / "classification_metrics.json"
    matrix_file = output / "confusion_matrix.csv"
    m = json.loads(metrics_file.read_text(encoding="utf-8"))
    RESULTS["Model analysis (ZIP)"] = (metrics_file, matrix_file)
    print(f"Model: {ANALYSIS_MODEL}")
    print(f"Test images scored: {m['n_scored']} (score threshold: {m['score_threshold']:g})")
    print(f"Actual class totals: {m['tn'] + m['fp']} negative, "
          f"{m['tp'] + m['fn']} positive.")
    print(f"Test accuracy: {m['accuracy']:.1%}")

    print("Confusion matrix (raw Test counts):")
    display(pd.DataFrame(
        {
            "Positive result": [m["tp"], m["fp"]],
            "Negative result": [m["fn"], m["tn"]],
        },
        index=["Has autism", "Does not have autism"],
    ))

    false_positive_rate = m["fp"] / (m["fp"] + m["tn"])
    true_negative_rate = m["specificity"]
    false_negative_rate = m["fn"] / (m["fn"] + m["tp"])
    true_positive_rate = m["sensitivity"]

    print(f"False positive rate: {false_positive_rate:.1%}")
    print(f"True negative rate:  {true_negative_rate:.1%}")
    print(f"False negative rate: {false_negative_rate:.1%}")
    print(f"True positive rate:  {true_positive_rate:.1%}")

    prevalence = 0.032
    tpr = true_positive_rate
    tnr = true_negative_rate

    joint = pd.DataFrame(
        {
            "Positive result": [
                prevalence * tpr,
                (1 - prevalence) * (1 - tnr)
            ],
            "Negative result": [
                prevalence * (1 - tpr),
                (1 - prevalence) * tnr
            ]
        },
        index=["Has autism", "Does not have autism"]
    )

    print("Share of the whole population in each outcome:")
    display((joint * 100).round(2))

    print("Chance of actual status, given the result:")
    display((joint / joint.sum(axis=0) * 100).round(2))

    print("Download these results in Cell 7.")


if RUN_MODEL_ANALYSIS:
    invalidate_bayesian_controls()
    if ANALYSIS_MODEL == "Retrained in this runtime":
        RETRAINED_BAYES_METRICS = None
    repo = project_root()
    runtime_python()
    saved_model = ANALYSIS_MODEL == "Saved project model"
    artifacts = SAVED if saved_model else globals().get("TRAINED")
    if not saved_model and (not artifacts or not (artifacts / "model.pt").is_file()):
        raise RuntimeError("Choose the saved project model or retrain in Cell 3.")
    analyzed_model_id = None if saved_model else bayes_model_identity(ANALYSIS_MODEL)
    flags = ["--saved-project-model"] if saved_model else []
    if RUN_PIXEL_SENSITIVITY:
        flags.append("--include-heatmap")

    fixed_test = bundled_test_archive()
    RESULTS.pop("Model analysis (ZIP)", None)
    analysis_root = repo / "outputs" / "model_analysis"
    analysis_root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run-", dir=analysis_root))

    with tempfile.TemporaryDirectory(prefix="evaluation-", dir=WORK) as folder:
        stage = Path(folder)
        extracted, crops = stage / "extracted", stage / "crops"
        checked_extract(fixed_test, extracted)
        run_script("test_inputs", "cropped-test", extracted, crops)
        run_script("analyze", "test", artifacts, crops, output, *flags)

    display_test_results(output)
    if analyzed_model_id is not None:
        RETRAINED_BAYES_METRICS = (analyzed_model_id, output / "classification_metrics.json")
    if RUN_PIXEL_SENSITIVITY:
        heatmap = output / "pixel_sensitivity.png"
        RESULTS["Model analysis (ZIP)"] += (heatmap,)
        display(Image(filename=str(heatmap)))
else:
    print("Model analysis skipped.")

## Cell 5: Try the model on photos

Choose the saved or retrained model. Then choose **Individual photos** to select one or more files, or **ZIP of a photo folder** to upload a whole batch as one ZIP. Run the cell and select your files when prompted.

Use photos you have permission to upload. Each photo should show **one child only**, with their **whole face visible**, both eyes open, and no glasses or hair over the eyes. The child doesn’t need to face the camera directly. If there are other people in the photo, crop them out first while keeping the child’s full face visible. The program will make the smaller crop around the eyes.

The cell reports how many photos it scored and how many it labeled autistic. If you submitted one photo and it was scored, **1 positive** means the model predicted autistic and **0 positives** means it predicted non-autistic. **0 photos scored** means no prediction was made. Try another photo if that happens. For one row per scored photo, download **Predictions CSV** in **Cell 7**.


In [ ]:
# @title Cell 5. Score one or more photos
RUN_PREDICTION = True  # @param {type:"boolean"}
MODEL_SOURCE = "Saved project model"  # @param ["Saved project model", "Retrained in this runtime"]
SHOW_PREDICTION_LOG = True  # @param {type:"boolean"}
PHOTO_SOURCE = "Individual photos"  # @param ["Individual photos", "ZIP of a photo folder"]

import csv
from zipfile import BadZipFile
from IPython.display import clear_output


MAX_PHOTO_ZIP_ENTRIES = 10_000
MAX_PHOTO_BYTES = 32 * 1024**2
MAX_PHOTO_ZIP_BYTES = 512 * 1024**2


def stage_individual_photos(uploads, raw):
    """Stage each selected photo separately so crop names cannot collide."""
    uploads.mkdir()
    print("Choose one photo or select several photos inside a folder.")
    uploaded = files.upload(target_dir=str(uploads))
    names = list(uploaded)
    del uploaded
    paths = []
    for name in names:
        path = Path(name)
        # Colab may return either a full path or a basename.
        if not path.resolve().is_relative_to(uploads):
            path = uploads / path
        path = path.resolve()
        if (not path.is_relative_to(uploads) or not path.is_file()
                or path.suffix.lower() not in PHOTO_TYPES):
            raise ValueError("Choose supported image files only (JPG, PNG, TIFF, BMP or WebP). "
                             "For an archive, use ZIP of a photo folder.")
        paths.append(path)
    if not paths:
        raise ValueError("No photos were selected.")

    source_names = {}
    for i, path in enumerate(sorted(paths)):
        slot = f"{i:06d}"
        staged = raw / slot / path.name
        staged.parent.mkdir(parents=True)
        shutil.copyfile(path, staged)
        source_names[f"{slot}/{path.stem}_eyes.jpg"] = path.relative_to(uploads).as_posix()
    return source_names


def stage_zipped_photos(archive, raw):
    """Stage only supported photos and preserve their relative ZIP paths in the CSV."""
    if not archive.is_file():
        raise ValueError("No ZIP was selected.")
    if archive.stat().st_size > MAX_PHOTO_ZIP_BYTES:
        raise ValueError("The photo ZIP exceeds 512 MiB. Split it into smaller batches.")

    try:
        with ZipFile(archive) as z:
            entries = z.infolist()
            if len(entries) > MAX_PHOTO_ZIP_ENTRIES:
                raise ValueError("The ZIP has more than 10,000 entries. Split it into smaller batches.")

            photos = {}
            total = 0
            for entry in entries:
                path = PurePosixPath(entry.filename.replace(chr(92), "/"))
                if (entry.is_dir() or path.suffix.lower() not in PHOTO_TYPES
                        or any(part.startswith(".") or part == "__MACOSX" for part in path.parts
                               if part != "..")):
                    continue
                mode = stat.S_IFMT(entry.external_attr >> 16)
                if (path.is_absolute() or not path.parts or ".." in path.parts
                        or len(entry.filename) > 1024
                        or any(":" in part for part in path.parts)
                        or any(ord(char) < 32 for char in entry.orig_filename)
                        or mode not in (0, stat.S_IFREG)):
                    raise ValueError("The ZIP contains a photo with an unsafe path or file type.")
                if path in photos:
                    raise ValueError("The ZIP repeats a photo's relative path. Remove the duplicate entry.")
                if entry.flag_bits & 1:
                    raise ValueError("The ZIP contains encrypted photos. Upload an unencrypted ZIP.")
                total += entry.file_size
                if entry.file_size > MAX_PHOTO_BYTES or total > MAX_PHOTO_ZIP_BYTES:
                    raise ValueError("Photo ZIP limits are 32 MiB per photo and 512 MiB total expanded.")
                photos[path] = entry

            if not photos:
                raise ValueError("No supported photos were found in the ZIP.")

            source_names = {}
            for i, (path, entry) in enumerate(sorted(photos.items())):
                slot = f"{i:06d}"
                target = raw / slot / path.name
                target.parent.mkdir(parents=True)
                with z.open(entry) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
                source_names[f"{slot}/{path.stem}_eyes.jpg"] = path.as_posix()
            return source_names
    except (BadZipFile, NotImplementedError) as error:
        raise ValueError("Choose a readable ZIP of a photo folder.") from error


# Clear stale counts before any new attempt, including a skipped or failed run.
PREDICTION_COUNTS = None
PREDICTION_MODEL_ID = None
invalidate_bayesian_controls()
globals().get("RESULTS", {}).pop("Predictions CSV", None)

if RUN_PREDICTION:
    repo = project_root()
    runtime_python()
    if MODEL_SOURCE == "Saved project model":
        artifacts = SAVED
        flags = ["--trusted-legacy-model"]
    else:
        if not TRAINED or not (TRAINED / "model.pt").is_file():
            raise RuntimeError("First retrain a model in Cell 3.")
        artifacts = TRAINED
        flags = []

    scored_model_id = bayes_model_identity(MODEL_SOURCE)
    output = repo / "outputs" / "prediction" / "predictions.csv"
    headers = ["image", "class_1_probability", "predicted_class", "source_image"]

    with tempfile.TemporaryDirectory(prefix="photos-", dir=WORK) as folder:
        stage = Path(folder).resolve()
        uploads, raw, crops = (stage / name for name in ("uploads", "raw", "crops"))
        if PHOTO_SOURCE == "Individual photos":
            source_names = stage_individual_photos(uploads, raw)
        elif PHOTO_SOURCE == "ZIP of a photo folder":
            archive = stage / "photos.zip"
            print("Choose one ZIP containing a folder of photos.")
            files.upload_file(str(archive))
            source_names = stage_zipped_photos(archive, raw)
        else:
            raise ValueError("Choose one of the listed photo sources.")
        selected_count = len(source_names)

        run_eye_crop(raw, crops, show_log=SHOW_PREDICTION_LOG)
        rows = []
        if any(crops.rglob("*_eyes.jpg")):
            scores = stage / "scores.csv"
            run_script("predict", crops, "--cropped", "--artifacts-dir", artifacts,
                       "--output", scores, *flags, private=not SHOW_PREDICTION_LOG)
            with scores.open(encoding="utf-8", newline="") as stream:
                rows = list(csv.DictReader(stream))
            for row in rows:
                row["source_image"] = source_names[row["source_image"]]

        output.parent.mkdir(parents=True, exist_ok=True)
        with output.open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=headers)
            writer.writeheader()
            writer.writerows(rows)

    clear_output(wait=True)
    count = len(rows)
    positive_count = sum(int(row["predicted_class"]) == 1 for row in rows)
    RESULTS["Predictions CSV"] = (output,)
    if count:
        PREDICTION_COUNTS = (count, positive_count)
        PREDICTION_MODEL_ID = scored_model_id
    print(f"Scored {count} of {selected_count} selected photos.")
    print(f"Predicted positive: {positive_count} out of {count} scored photos.")
    if count:
        print("Cell 6 pre-fills these counts when the same model is selected.")
    else:
        print("No usable eye strips were found; the CSV contains only its header.")
    print("Download the CSV in Cell 7.")
else:
    print("Prediction skipped.")

## Cell 6: Explore results from several photos

Run the cell to open the Bayesian calculator and choose which model to use:

- **Saved project model:** The calculator uses the confusion matrix included with the project. You can skip Cell 4.
- **Retrained in this runtime:** Run **Cell 4** with **Retrained in this runtime** selected first. The calculator needs the test results for that same model. If you train another model, run Cell 4 again.

If **Cell 5** scored photos using the model you select here, the calculator starts with that run’s total images and predicted positives. Otherwise, it starts with an example: **12 positives out of 20 images**. You can edit both counts. You can also change the **assumed prevalence**, the share of the population you assume is autistic. For example, enter `0.032` for 3.2%.

If you’re exploring results from uploaded photos, use photos of **one child at a time**. Different photos may add information that a single photo misses, but similar photos may lead to the same mistake. The calculator assumes the predictions are independent once the true label is known. If that assumption doesn’t hold, the result can look more certain than it should. Treat it as an example of how the calculation works, not as a personal probability or a diagnosis.

Click **Run calculation** to see the result.


In [ ]:
# @title Cell 6. Run the example from hypothetical counts
RUN_BAYES_EXAMPLE = True  # @param {type:"boolean"}

import csv
import json
import math
from IPython.display import Image, display


def retrained_test_counts():
    identity = bayes_model_identity("Retrained in this runtime")
    evaluation = globals().get("RETRAINED_BAYES_METRICS")
    hint = "Run Cell 4 with Retrained in this runtime selected before calculating."
    if not evaluation or evaluation[0] != identity:
        raise RuntimeError(hint)
    try:
        metrics = json.loads(evaluation[1].read_text(encoding="utf-8"))
        counts = tuple(metrics[k] for k in ("tn", "fp", "fn", "tp"))
        tn, fp, fn, tp = counts
        if (metrics.get("split") != "Test"
                or any(type(n) is not int or n < 0 for n in counts)
                or sum(counts) != metrics.get("n_scored")
                or not (tn + fp and fn + tp)):
            raise ValueError("Expected held-out counts for both classes.")
    except (OSError, KeyError, TypeError, ValueError) as error:
        raise RuntimeError(hint) from error
    return counts


def calculate_bayesian_example(n_images, n_positives, prevalence, show_plot, model_source):
    repo = project_root()
    if n_images < 1 or not 0 <= n_positives <= n_images:
        raise ValueError("Enter at least one image and a positive count from zero to that total.")
    if not 0 < prevalence < 1:
        raise ValueError("Enter prevalence as a fraction strictly between 0 and 1.")

    # Use reader-facing labels for the calculation plots.
    script = repo / "scripts" / "bayes.py"
    original = script.read_text(encoding="utf-8")
    updated = original
    labels = {
        'Given a positive reference label': 'Given the person has autism',
        'Given a negative reference label': 'Given the person does not have autism',
        'Hypothetical posterior': 'Probability of autism given the results',
        'ylabel="Posterior (%)"': 'ylabel="Probability of autism (%)"',
        'orientation="horizontal", label="Posterior fraction")': (
            'orientation="horizontal",\n'
            '                 label="Probability of autism given that many positives",\n'
            '                 ticks=[0, 0.25, 0.5, 0.75, 1],\n'
            '                 format=lambda value, _: f"{value:.0%}")'
        ),
    }
    for old_label, new_label in labels.items():
        updated = updated.replace(old_label, new_label)
    if updated != original:
        script.write_text(updated, encoding="utf-8")

    out = repo / "outputs" / "bayes"
    with tempfile.TemporaryDirectory(prefix="bayes-matrix-", dir=WORK) as folder:
        if model_source == "Saved project model":
            matrix = SAVED / "confusion_matrix.joblib"
            rates_source = "Bundled confusion matrix"
        elif model_source == "Retrained in this runtime":
            tn, fp, fn, tp = retrained_test_counts()
            matrix = Path(folder) / "confusion_matrix.joblib"
            # Use the same Python environment as the existing Bayesian script.
            run_command([runtime_python(), "-c",
                         "import joblib, json, sys; "
                         "joblib.dump(json.loads(sys.argv[2]), sys.argv[1])",
                         matrix, json.dumps([[tn, fp], [fn, tp]])])
            rates_source = "Held-out Test results from Cell 4 for the retrained model"
        else:
            raise ValueError("Choose one of the listed models.")
        print(f"Model: {model_source}. Rates: {rates_source}.")
        run_script(
            "bayes", "--confusion-matrix", matrix,
            "--n-images", n_images,
            "--positive-count", n_positives,
            "--prevalence", prevalence,
            "--color-scale", "linear",
            "--output-dir", out,
        )

    with (out / "bayes.csv").open(encoding="utf-8", newline="") as stream:
        selected = next(
            row for row in csv.DictReader(stream)
            if int(row["positive_count"]) == n_positives
        )
    posterior = float(selected["hypothetical_posterior"])
    probability = f"{posterior:.2%}" if math.isfinite(posterior) else "undefined"
    print(
        f"Hypothetical probability of autism given "
        f"{n_positives} positives out of {n_images}: "
        f"{probability} (with {prevalence:.2%} assumed prevalence)."
    )
    print("This is a model-based example, not a diagnosis.")
    print("Run Cell 7 to download the Bayesian example ZIP.")

    RESULTS["Bayesian example (ZIP)"] = (out / "bayes.csv", out / "bayes.png")
    if show_plot:
        display(Image(filename=str(out / "bayes.png")))


def show_bayesian_controls():
    import ipywidgets as widgets

    counts = globals().get("PREDICTION_COUNTS")
    prediction_id = globals().get("PREDICTION_MODEL_ID")
    generation = globals().get("BAYES_INPUT_GENERATION")
    saved_label = "Saved project model"
    retrained_label = "Retrained in this runtime"
    default_model = saved_label
    if counts and prediction_id:
        try:
            if prediction_id == bayes_model_identity(prediction_id[0]):
                default_model = prediction_id[0]
        except (OSError, RuntimeError, ValueError):
            pass

    style = {"description_width": "initial"}
    model = widgets.Dropdown(options=[saved_label, retrained_label], value=default_model,
                             description="Model:", style=style)
    images = widgets.IntText(value=20, description="Total images:", style=style)
    positives = widgets.IntText(value=12, description="Predicted positives:", style=style)
    prevalence = widgets.FloatText(value=0.032, description="Assumed prevalence (fraction):", style=style)
    plot = widgets.Checkbox(value=True, description="Show plot")
    run = widgets.Button(description="Run calculation", button_style="primary")
    source_note = widgets.HTML()
    rate_note = widgets.HTML()
    output = widgets.Output()
    displayed_identity = None

    def update_model(_=None):
        nonlocal displayed_identity
        if globals().get("BAYES_INPUT_GENERATION") is not generation:
            source_note.value = "An earlier cell has run again. Rerun Cell 6 to refresh."
            return
        with output:
            output.clear_output(wait=False)
        displayed_identity = None
        try:
            displayed_identity = bayes_model_identity(model.value)
        except (OSError, RuntimeError, ValueError):
            pass
        use_counts = bool(counts and displayed_identity is not None and prediction_id == displayed_identity)
        images.value, positives.value = counts if use_counts else (20, 12)
        source_note.value = (
            "Prefilled from Cell 5 for this exact model. Both counts are editable."
            if use_counts else
            "No matching scored predictions from Cell 5. Defaults: 20 images, 12 positives. Both are editable."
        )
        if model.value == saved_label:
            rate_note.value = "Uses the confusion matrix bundled with the saved model."
        else:
            try:
                retrained_test_counts()
            except (OSError, RuntimeError):
                rate_note.value = "Run Cell 4 for the current retrained model to enable this calculation."
            else:
                rate_note.value = "Uses Cell 4’s held-out Test results for this retrained model."

    def on_run(_):
        with output:
            output.clear_output(wait=True)
            if globals().get("BAYES_INPUT_GENERATION") is not generation:
                print("Cell 1, 3, 4 or 5 has run again. Rerun Cell 6 to refresh.")
                return
            run.disabled = True
            globals().get("RESULTS", {}).pop("Bayesian example (ZIP)", None)
            try:
                if displayed_identity is not None and displayed_identity != bayes_model_identity(model.value):
                    raise RuntimeError("The model artifacts changed. Rerun Cell 6 to refresh.")
                calculate_bayesian_example(
                    images.value, positives.value, prevalence.value, plot.value, model.value
                )
            except (OSError, ValueError, RuntimeError, subprocess.CalledProcessError) as error:
                print(f"Could not calculate: {error}")
            finally:
                run.disabled = False

    model.observe(update_model, names="value")
    run.on_click(on_run)
    update_model()
    display(widgets.VBox([model, source_note, rate_note, images, positives,
                          prevalence, plot, run, output]))


if RUN_BAYES_EXAMPLE:
    project_root()
    show_bayesian_controls()
else:
    print("Bayesian example skipped.")

## Cell 7: Download your results

Choose the result you want, check the download box, and run the cell. Run the corresponding cell before trying to download its files.

- **Predictions CSV:** one row per photo scored in Cell 5. A CSV opens in a spreadsheet app such as Excel.
- **Model analysis (ZIP):** the test-set results and the heatmap if you generated one in Cell 4.
- **Bayesian example (ZIP):** the table of calculated probabilities and the plot from Cell 6.
- **Retrained model and training log (ZIP):** the model and training files from Cell 3.

Download anything you want to keep before ending your Colab session.


In [ ]:
# @title Cell 7. Download one result
DOWNLOAD_SELECTED = False  # @param {type:"boolean"}
RESULT_TO_DOWNLOAD = "Predictions CSV"  # @param ["Predictions CSV", "Bayesian example (ZIP)", "Retrained model and training log (ZIP)", "Model analysis (ZIP)"]

if DOWNLOAD_SELECTED:
    project_root()
    selected = RESULTS.get(RESULT_TO_DOWNLOAD)
    if not selected or any(not item.is_file() for item in selected):
        raise FileNotFoundError("Run the selected workflow before downloading a result.")
    if RESULT_TO_DOWNLOAD == "Predictions CSV":
        files.download(str(selected[0]))
    else:
        names = {
            "Bayesian example (ZIP)": "bayesian-example.zip",
            "Retrained model and training log (ZIP)": "retrained-model-and-training-log.zip",
            "Model analysis (ZIP)": "model-analysis.zip",
        }
        name = names[RESULT_TO_DOWNLOAD]
        package = WORK / name
        with ZipFile(package, "w", compression=ZIP_DEFLATED) as archive:
            for item in selected:
                archive.write(item, arcname=item.name)
        files.download(str(package))
else:
    print("Download skipped.")